## Exercise: The Random Forest

#### In this exercise, we build, evaluate, and compare random forest regression models.

### Learning Objectives
#### By the end of this train, you should be able to:
##### - Build a random forest regression model in Python
##### - Experiment with different numbers of trees
##### - Evaluate feature importance using a random forest

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

In [2]:
# Load dataset
df = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/Python/Crop_yield.csv')
df.head(5)

,Region,Temperature,Rainfall,Soil_Type,Fertilizer_Usage,Pesticide_Usage,Irrigation,Crop_Variety,Yield
0,East,23.152156,803.362573,Clayey,204.792011,20.767590,1,Variety B,40.316318
1,West,19.382419,571.567670,Sandy,256.201737,49.290242,0,Variety A,26.846639
2,North,27.895890,-8.699637,Loamy,222.202626,25.316121,0,Variety C,-0.323558
3,East,26.741361,897.426194,Loamy,187.984090,17.115362,0,Variety C,45.440871
4,East,19.090286,649.384694,Loamy,110.459549,24.068804,1,Variety B,35.478118


### Preparing the Dataset

In [3]:
# Dummy variable encoding for categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

In [4]:
df_encoded.head()

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_North,Region_South,Region_West,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_Variety B,Crop_Variety_Variety C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,0,0,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,0,0,1,0,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,1,0,0,1,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,0,0,0,1,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,0,0,0,1,0,1,0


### Exercise 1

#### Create a function named `train_rf_model` to train and evaluate a random forest regression model on the encoded dataset

#### The function should take in three parameters:
##### - A DataFrame containing the encoded features
##### - A string containing the name of the target variable
##### - The number of estimators for the random forest

#### It the returns:
##### - The trained model object
##### - The RMSE and R² scores of the model's performance on the test set

In [5]:
def train_rf_model(df, target_var, num_estimators):
    X = df.drop(target_var, axis=1)
    y = df[target_var]
    
#     X = X.reshape(-1, 1)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42
    )
    
    forest = RandomForestRegressor(
        n_estimators=num_estimators,
        random_state=42
    )
    
    forest.fit(X_train, y_train)
    
    y_pred = forest.predict(X_test)
    
    rmse = np.sqrt(metrics.mean_squared_error(y_test, y_pred))
    r2 = metrics.r2_score(y_test, y_pred)
    
    return forest, {'RMSE': rmse, 'R2': r2}

### Exercise 2

#### Use the function you have defined in **Exercise 1** to train and evaluate three different random forest regression models with each having the following number of estimators respectively: `50`, `100`, `200`. Store the results in a dictionary

In [8]:
# model_1 = train_rf_model(df_encoded, 'Yield', 50)
# model_2 = train_rf_model(df_encoded, 'Yield', 100)
# model_3 = train_rf_model(df_encoded, 'Yield', 200)

# Examiner approach
estimators_list = [50, 100, 200]

results = {}

for n in estimators_list:
    model, metric = train_rf_model(df_encoded, 'Yield', n)
    results[f'{n} trees'] = metric
    
results

{'50 trees': {'RMSE': 0.9188430167365879, 'R2': 0.990924630174936},
 '100 trees': {'RMSE': 0.9110424449326404, 'R2': 0.9910780678605957},
 '200 trees': {'RMSE': 0.9228447177589202, 'R2': 0.9908454088087382}}

### Exercise 3

#### Say we wish to understand which features have the most impact on crop yield predictions. Use the `feature_importances_` attribute from our last trained random forest model in **Exercise 2** to return a series containing the feature importance score for each of the features in our dataset, sorted in descending order

In [13]:
feature_importances = model.feature_importances_

feature_names = df_encoded.drop('Yield', axis=1).columns

importances = pd.Series(feature_importances, index=feature_names).sort_values(ascending=False)

importances

Rainfall                  0.978758
Fertilizer_Usage          0.016789
Temperature               0.001974
Pesticide_Usage           0.001127
Irrigation                0.000261
Region_West               0.000206
Crop_Variety_Variety B    0.000200
Soil_Type_Loamy           0.000158
Soil_Type_Sandy           0.000147
Region_North              0.000135
Crop_Variety_Variety C    0.000127
Region_South              0.000117
dtype: float64